# Notebook 33 — What does the SSL latent encode (if not MJO phase)?
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment
**Author:** Jiayi (jh9141@nyu.edu)

The SSL-nb15 latent has strong ENSO content (z~18) but its angle is NOT MJO phase (circ_corr 0.17). What
*is* it organizing by? **Key framing:** the input `X_MJO_bp20_90` is 20-90-day band-passed, so the slow
ENSO/seasonal signal is filtered out of the input -> the latent's "ENSO" must be the **ENSO-modulated
intraseasonal MJO structure** (amplitude / convection longitude / R-K asymmetry differ by ENSO).
Hypothesis: the latent encodes the **MJO state** (amplitude + convection location + character), ENSO-decodable
via that structure, phase weak as an angle.

Tools (per Session-55 plan):
1. **Linear + MLP probes** (frozen latent) for phase, amplitude, ENSO, convection-longitude, month.
   linear-vs-MLP gap = linear vs nonlinear storage.
2. **Direction-regression** (geometry: which latent axis = which variable).
3. **Latent power spectrum** (label-free: ~30-90 d peak = phase vs slower = envelope).
4. **Harder physical targets** (Zhang 2020): Maritime-Continent crossing / propagating-vs-non-propagating,
   life-cycle stage, Rossby-Kelvin asymmetry.

Primary latent = SSL-nb15; `LATENTS` dict lets you add aux2d/aux3d/NSV/Barlow later.

---

## Cell 1 — Load SSL latent + labels + fields + build targets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, r2_score

PROJECT_DIR='/content/drive/MyDrive/BSISO_SSL_Project'; MJO_DIR=f'{PROJECT_DIR}/MJO'
PROC=f'{MJO_DIR}/data/processed'; OUT=f'{MJO_DIR}/moisture_constraints/results/latent_content'
os.makedirs(OUT, exist_ok=True)

LATENTS = {'SSL-nb15': f'{MJO_DIR}/results/ssl/embeddings.npy'}   # add aux2d/aux3d/NSV/Barlow later
NAME='SSL-nb15'; Z=np.load(LATENTS[NAME]).astype(np.float32)     # (M,2) on bp axis

labels=pd.read_csv(f'{PROC}/labels_aligned_mjo_bp20_90.csv', parse_dates=['date'])
X_bp=np.load(f'{PROC}/X_MJO_bp20_90.npy'); lons=np.load(f'{PROC}/longitudes_mjo.npy')
M=len(Z); assert len(labels)==M==len(X_bp)
dates=pd.DatetimeIndex(labels['date']).normalize()
u850=X_bp[:,0,0,:]; olr=X_bp[:,1,0,:]                            # ch0 u850', ch1 OLR'

phase=labels['phase'].values.astype(int); amp=labels['amplitude'].values
enso=labels['enso_category'].values; weak=labels['weak_mjo'].values.astype(bool)
active=(~weak)&(amp>=1.0)
enso_num=np.array([{'El Nino':1.0,'Neutral':0.0,'La Nina':-1.0}[c] for c in enso])
month=dates.month.values
conv_lon=lons[np.argmin(olr,axis=1)].astype(float)               # longitude of strongest convection (min OLR')
ph_rad=(phase-1)/8.0*2*np.pi

# own-RMM mapped to bp axis (reference phase + amplitude)
full=pd.read_csv(f'{PROC}/labels_aligned_mjo.csv', parse_dates=['date'])
fr={d:i for i,d in enumerate(pd.DatetimeIndex(full['date']).normalize())}
fi=np.array([fr.get(d,-1) for d in dates])
pcs=np.load(f'{PROC}/mjo_rmm_own_pcs.npy'); pcs_bp=np.where(fi[:,None]>=0,pcs[np.clip(fi,0,None)],np.nan)

years=dates.year.values; val_years=set(sorted(np.unique(years))[::5])
is_val=np.isin(years,list(val_years)); tr=~is_val; va=is_val
print(f'{NAME}: Z {Z.shape}  active {int(active.sum())}  val years {sorted(val_years)[:4]}...')

## Cell 2 — Linear + MLP probes (decodability; linear vs nonlinear storage)

For each target: a **linear** probe and a small **MLP** probe from the frozen latent (year-based split).
The linear-vs-MLP gap shows whether the information is stored linearly or nonlinearly (e.g. folded phase).

In [ ]:
sc=StandardScaler().fit(Z[tr]); Zs=sc.transform(Z)
def clf_probe(y, mask, balanced=False):
    m_tr=tr&mask; m_va=va&mask
    lin=LogisticRegression(max_iter=2000, C=1.0, class_weight=('balanced' if balanced else None))
    lin.fit(Zs[m_tr],y[m_tr]); pl=lin.predict(Zs[m_va])
    mlp=MLPClassifier(hidden_layer_sizes=(64,32), max_iter=400, random_state=0)
    mlp.fit(Zs[m_tr],y[m_tr]); pm=mlp.predict(Zs[m_va])
    sc_fn=balanced_accuracy_score if balanced else accuracy_score
    return float(sc_fn(y[m_va],pl)), float(sc_fn(y[m_va],pm)), int(m_va.sum())
def reg_probe(y, mask):
    m_tr=tr&mask&np.isfinite(y); m_va=va&mask&np.isfinite(y)
    lin=Ridge(alpha=1.0).fit(Zs[m_tr],y[m_tr])
    mlp=MLPRegressor(hidden_layer_sizes=(64,32),max_iter=600,random_state=0).fit(Zs[m_tr],y[m_tr])
    return float(r2_score(y[m_va],lin.predict(Zs[m_va]))), float(r2_score(y[m_va],mlp.predict(Zs[m_va]))), int(m_va.sum())

rows=[]
l,mp,n=clf_probe(phase, active);          rows.append(['MJO phase (8-class acc)', round(1/8,3), round(l,3), round(mp,3), n])
l,mp,n=reg_probe(amp, np.ones(M,bool));   rows.append(['amplitude (R2)', 0.0, round(l,3), round(mp,3), n])
l,mp,n=clf_probe(enso, np.ones(M,bool), balanced=True); rows.append(['ENSO (3-class bal-acc)', round(1/3,3), round(l,3), round(mp,3), n])
l,mp,n=reg_probe(conv_lon, active);       rows.append(['convection longitude (R2)', 0.0, round(l,3), round(mp,3), n])
l,mp,n=clf_probe(month, np.ones(M,bool)); rows.append(['calendar month (12-class acc)', round(1/12,3), round(l,3), round(mp,3), n])
probe_df=pd.DataFrame(rows, columns=['target','baseline','linear','MLP','n_val'])
print(probe_df.to_string(index=False)); probe_df.to_csv(f'{OUT}/{NAME}_probes.csv',index=False)

fig,ax=plt.subplots(figsize=(11,4.5)); xs=np.arange(len(rows))
ax.bar(xs-0.2, probe_df['linear'], 0.38, label='linear', color='#1f77b4')
ax.bar(xs+0.2, probe_df['MLP'],    0.38, label='MLP',    color='#ff7f0e')
ax.plot(xs, probe_df['baseline'], 'k_', ms=18, label='chance/baseline')
ax.set_xticks(xs); ax.set_xticklabels(probe_df['target'], rotation=20, ha='right', fontsize=8)
ax.set_ylabel('val accuracy / R2'); ax.set_title(f'{NAME}: what is linearly vs nonlinearly decodable?'); ax.legend()
plt.tight_layout(); p=f'{OUT}/{NAME}_probes.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 3 — Direction-regression (geometry: which latent axis is which variable)

In [ ]:
Zc=Z-Z.mean(0)
def direction(y, mask):
    m=mask&np.isfinite(y); r=Ridge(alpha=1.0).fit(Zc[m],y[m]); w=r.coef_
    return w/ (np.linalg.norm(w)+1e-9), float(r2_score(y[m], r.predict(Zc[m])))
targets={'amplitude':(amp,np.ones(M,bool)),'ENSO':(enso_num,np.ones(M,bool)),
         'conv_lon':(conv_lon,active),'phase_cos':(np.cos(ph_rad),active),'phase_sin':(np.sin(ph_rad),active)}
dirs={};
for k,(y,msk) in targets.items():
    w,r2=direction(y,msk); dirs[k]=(w,r2); print(f'{k:10s} direction {w.round(2)}  R2={r2:.3f}')

fig,ax=plt.subplots(figsize=(6,6))
sub=np.random.default_rng(0).choice(np.where(active)[0],min(3000,int(active.sum())),replace=False)
ax.scatter(Zc[sub,0],Zc[sub,1],s=5,alpha=.15,color='gray')
col={'amplitude':'green','ENSO':'red','conv_lon':'purple','phase_cos':'navy','phase_sin':'teal'}
sc_=np.percentile(np.abs(Zc),95)
for k,(w,r2) in dirs.items():
    ax.arrow(0,0,w[0]*sc_,w[1]*sc_,head_width=sc_*0.04,color=col[k],lw=2,length_includes_head=True)
    ax.text(w[0]*sc_*1.1,w[1]*sc_*1.1,f'{k}\nR2={r2:.2f}',color=col[k],fontsize=8,ha='center')
ax.set_aspect('equal'); ax.set_title(f'{NAME}: physical-variable directions in latent plane')
ax.axhline(0,color='k',lw=.3); ax.axvline(0,color='k',lw=.3)
plt.tight_layout(); p=f'{OUT}/{NAME}_directions.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)

## Cell 4 — Latent power spectrum (label-free: fast phase vs slow envelope)

Welch periodogram of the latent on the chronological bp axis: `z1`, complex `z1+i*z2` (eastward vs
westward), and `|z|` (amplitude envelope), vs the own-RMM spectrum. A sharp ~30-90 d peak = a propagating
phase signal; broadband/slow power = an amplitude/activity envelope.

In [ ]:
from scipy.signal import welch
order=np.argsort(dates.values); Zo=Z[order]; pcs_o=pcs_bp[order]
def spec(x, fs=1.0, nper=512):
    f,P=welch(x-np.nanmean(x), fs=fs, nperseg=min(nper,len(x)), detrend='linear')
    return f[1:], P[1:]
f1,P1=spec(Zo[:,0]); fz,Pz=spec(Zo[:,0]+1j*Zo[:,1] if False else np.hypot(Zo[:,0],Zo[:,1]))   # |z| envelope
fc,Pc=welch(Zo[:,0]-Zo[:,0].mean()+1j*(Zo[:,1]-Zo[:,1].mean()), fs=1.0, nperseg=512, detrend=False, return_onesided=False)
fr_,Pr=spec(pcs_o[:,0]+0.0)   # own-RMM PC1 (finite)

fig,ax=plt.subplots(1,2,figsize=(14,4.5))
for ff,PP,lab,c in [(f1,P1,'SSL z1','#1f77b4'),(fz,Pz,'SSL |z| (envelope)','#ff7f0e'),(fr_,Pr,'own-RMM PC1','k')]:
    ok=ff>0; ax[0].plot(1/ff[ok], PP[ok]/np.nanmax(PP[ok]), label=lab, color=c)
ax[0].axvspan(30,90,color='green',alpha=.08,label='MJO 30-90 d'); ax[0].set_xscale('log'); ax[0].set_xlim(8,400)
ax[0].set_xlabel('period (days)'); ax[0].set_ylabel('norm. power'); ax[0].set_title(f'{NAME} latent vs own-RMM spectra'); ax[0].legend(fontsize=8)
# complex spectrum: eastward (f>0) vs westward (f<0) power of z1+i z2
per=np.where(fc!=0,1/np.abs(fc),np.inf); east=fc>0
ax[1].plot(per[east & (per<400)], Pc[east & (per<400)].real,'b.',ms=3,label='eastward')
ax[1].plot(per[~east & (per<400)], Pc[~east & (per<400)].real,'r.',ms=3,label='westward')
ax[1].axvspan(30,90,color='green',alpha=.08); ax[1].set_xscale('log'); ax[1].set_xlabel('period (days)')
ax[1].set_title('SSL z1+i*z2: eastward vs westward power'); ax[1].legend(fontsize=8)
plt.tight_layout(); p=f'{OUT}/{NAME}_spectrum.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)
# fraction of latent power in the 30-90d band
band=(1/f1>=30)&(1/f1<=90); print(f'SSL z1 power in 30-90d band: {100*P1[band].sum()/P1.sum():.0f}%')

## Cell 5 — Harder physical targets (Zhang 2020): MC-crossing, life-cycle, R-K asymmetry

Detect contiguous active runs (amp>=1, consecutive days, >=20 d). Label each: **MC-crossing** (convection
goes from west of 100E to east of 150E) vs non-propagating; **life-cycle** thirds (onset/mature/decay).
Plus per-day **Rossby-Kelvin** u850 asymmetry. Then probe the latent for each.

In [ ]:
runs=[]; i=0
while i<M:
    if active[i]:
        j=i
        while j+1<M and active[j+1] and int((dates[j+1]-dates[j]).days)==1: j+=1
        if j-i+1>=20: runs.append((i,j))
        i=j+1
    else: i+=1
print(f'{len(runs)} active runs >=20 d')
mc_label=np.full(M,-1); lc_label=np.full(M,-1)
for (a,b) in runs:
    cl=conv_lon[a:b+1]
    crossing = (np.nanmin(cl[:max(3,(b-a)//3)])<100) and (np.nanmax(cl[-(max(3,(b-a)//3)):])>150)
    mc_label[a:b+1]= 1 if crossing else 0
    L=b-a+1; th=L//3
    lc_label[a:a+th]=0; lc_label[a+th:a+2*th]=1; lc_label[a+2*th:b+1]=2     # onset/mature/decay
# R-K per day
west=np.max(u850,axis=1); east=np.max(-u850,axis=1); rk=west/np.maximum(east,1e-6)

res=[]
mmc=mc_label>=0
if mmc.sum()>100 and len(np.unique(mc_label[mmc]))==2:
    l,mp,n=clf_probe(mc_label, mmc); base=max(np.bincount(mc_label[mmc]).max()/mmc.sum(),0.5)
    res.append(['MC-crossing (binary acc)', round(base,3), round(l,3), round(mp,3), n])
mlc=lc_label>=0
l,mp,n=clf_probe(lc_label, mlc); res.append(['life-cycle stage (3-class acc)', round(1/3,3), round(l,3), round(mp,3), n])
l,mp,n=reg_probe(rk, active);    res.append(['Rossby-Kelvin ratio (R2)', 0.0, round(l,3), round(mp,3), n])
hard_df=pd.DataFrame(res, columns=['target','baseline','linear','MLP','n_val'])
print(hard_df.to_string(index=False)); hard_df.to_csv(f'{OUT}/{NAME}_hard_targets.csv',index=False)
print(f'\nMC-crossing runs: {int((mc_label==1).any()) and (np.array([mc_label[a]==1 for a,b in runs]).sum())} crossing / {len(runs)} total')

## Cell 6 — Summary + interpretation

In [ ]:
summary={'latent':NAME,'n_active':int(active.sum()),
         'probes':probe_df.to_dict(orient='records'),
         'directions':{k:{'dir':[float(w[0]),float(w[1])],'R2':round(r2,3)} for k,(w,r2) in dirs.items()},
         'z1_power_30_90d_frac':round(float(P1[band].sum()/P1.sum()),3),
         'hard_targets':hard_df.to_dict(orient='records')}
json.dump(summary, open(f'{OUT}/{NAME}_content_summary.json','w'), indent=2, default=float)
print('Saved', f'{OUT}/{NAME}_content_summary.json')
print('\n=== READ-OUT ===')
print(probe_df.to_string(index=False)); print(); print(hard_df.to_string(index=False))
print('\nInterpretation guide: high amplitude/conv_lon/ENSO probes + low phase (esp. linear) + a |z| envelope')
print('spectrum (not a clean 45-d line) => the latent is an MJO-STATE (amplitude+location) representation,')
print('ENSO-decodable via structure, not a phase clock. MC-crossing decodable => it also learned event character.')

---
## Done!
Outputs in `MJO/moisture_constraints/results/latent_content/`. To add another latent, append its
embedding path to `LATENTS` (must be on the bp20_90 axis, or date-align like nb30 Cell 9) and re-run.

**Next:** repeat for aux2d / aux3d / NSV-7D / Barlow and compare (CKA); the contrast tells us which
representations are MJO-state vs slow-envelope vs phase.

---
*DDCS Project | jh9141@nyu.edu*